In [89]:
from social_groups.trialrunner.experiment import main_registry

In [90]:
from collections import defaultdict
from pathlib import Path
from social_groups.polars_values import MODEL_NAME_TO_LETTER_MAPPING

import src.social_groups.polars_columns as plc
import polars as pl
from social_groups.analyzer.io_operations import read_hydra_config, read_meta_config
from social_groups.directories import TRACK_FILE_NAME, TRACK_FILE_NAME_COMPRESSED
from social_groups.general.tracking import TrackEntry, iter_jsonl_zst
from social_groups.reporting.group_reply import (
    GroupReplyAggregator,
    MajorityVote,
)
from social_groups.reporting.parsing import (
    AnswerComparer,
    AnswerOptions,
    AnswerParser,
)
from social_groups.trialrunner.data_connectors.mmlu_pro_subset import (
    MMLUProSubsetConnector,
)
from social_groups.trialrunner.utils.hydra_config import MainConfig

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [91]:
data_connector = MMLUProSubsetConnector()

input_data = pl.DataFrame([p.model_dump() | {"question": data_connector.prepare_example(p).question} for p in
                           data_connector.iterate_data()])

In [92]:
parser = AnswerParser(AnswerOptions.letters_A_to_J)
group_reply = GroupReplyAggregator(MajorityVote())
comparer = AnswerComparer(
    AnswerOptions.letters_A_to_J, triple_underscore_handling="wrong"
)

DIRS = [
    Path(
        "/Users/philipp/Documents/Studium/Informatik/Masterthesis/Repository/results/multirun/final/heterogeneous_group_baseline_with_tool_usage/2026-03-05-03-53-56")
]


In [93]:
dfs = []
for experiment_dir in DIRS:
    data = defaultdict(list)
    info: dict[str, MainConfig] = {}
    for i, project_path in enumerate(sorted(experiment_dir.iterdir())):
        if not project_path.is_dir():
            continue
        run_id = i

        info[project_path] = read_hydra_config(project_path)
        run_meta_info = read_meta_config(project_path)

        for item in iter_jsonl_zst(project_path / TRACK_FILE_NAME_COMPRESSED):
            data[project_path].append(TrackEntry.model_validate(item))

    dfs.append(pl.DataFrame(
        [({"question": (x := d.model_dump())["input"]["question"]} |
          {"experiment_name": info[project].experiment.name,
           "model_name": MODEL_NAME_TO_LETTER_MAPPING[
               info[project].experiment.strategy.configuration.backend.model_name]} |
          {a: b for a, b in x["output"].items()})
         for project, datas in data.items() for d in datas],
    ).drop("used_input_tokens", "used_output_tokens", "history"))

data = pl.concat(dfs).join(input_data, on="question").drop("src", "category", "cot_content").rename(
    {"answer": "answer_string"})

In [94]:
data.with_columns(
    parser(pl.col("final_answer")).alias("PARSED_VALUE"),
    comparer(
        parser(pl.col("final_answer")),
        pl.col("answer_string"),
    ).alias(plc.is_correct)
).group_by("model_name").agg(pl.col(plc.is_correct).mean().alias(plc.accuracy))

model_name,accuracy
str,f64
"""M""",0.43
"""L""",0.3
"""H""",0.52
